# 10.1 Cross-Validation

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.01_cross-validation.ipynb)

Companion code for Section 10.1 of the book. Run the cells in order from top to bottom.

**Example 10.1.1** · bash

```bash
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    KFold, StratifiedKFold, RepeatedStratifiedKFold, LeaveOneOut,
    TimeSeriesSplit, GroupKFold, cross_val_score, cross_validate,
    learning_curve
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=100, random_state=42))
])

# ── Compare CV strategies ─────────────────────────────────────────
print('=== Cross-Validation Strategy Comparison ===')
print(f'{'Strategy':<28} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Folds':>6}')
print('='*68)

strategies = {
    'KFold(k=3)':             KFold(n_splits=3, shuffle=True, random_state=42),
    'KFold(k=5)':             KFold(n_splits=5, shuffle=True, random_state=42),
    'KFold(k=10)':            KFold(n_splits=10, shuffle=True, random_state=42),
    'StratifiedKFold(k=5)':   StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'Repeated(5x3)':          RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42),
    'Repeated(5x10)':         RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42),
}

for name, cv in strategies.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{name:<28} {scores.mean():>8.4f} {scores.std():>8.4f} {scores.min():>8.4f} {scores.max():>8.4f}
{len(scores):>6}')

# ── Multiple metrics in one call ──────────────────────────────────
print('\n=== Multiple Metrics via cross_validate ===')
cv_results = cross_validate(
    pipe, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=['accuracy','precision_macro','recall_macro','f1_macro','roc_auc'],
    return_train_score=True,
    n_jobs=-1
)

for metric in ['accuracy','precision_macro','recall_macro','f1_macro','roc_auc']:
    test_key  = f'test_{metric}'
    train_key = f'train_{metric}'
    test_mean  = cv_results[test_key].mean()
    train_mean = cv_results[train_key].mean()
    gap = train_mean - test_mean
    flag = ' ← OVERFIT' if gap > 0.05 else ''
    print(f'  {metric:<18}: train={train_mean:.4f}  test={test_mean:.4f}  gap={gap:.4f}{flag}')

# ── Learning curves ───────────────────────────────────────────────
train_sizes, train_scores, test_scores = learning_curve(
    pipe, X, y,
    train_sizes=np.linspace(0.1, 1.0, 15),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Cross-Validation Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
test_mean  = test_scores.mean(axis=1)
test_std   = test_scores.std(axis=1)

ax.plot(train_sizes, train_mean, 'b-o', ms=6, lw=2, label='Train AUC')
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='blue')
ax.plot(train_sizes, test_mean, 'r-o', ms=6, lw=2, label='CV AUC')
ax.fill_between(train_sizes, test_mean-test_std, test_mean+test_std, alpha=0.15, color='red')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('AUC Score')
ax.set_title('Learning Curve\n(Random Forest on Breast Cancer)')
ax.legend(); ax.grid(alpha=0.3)

# CV strategy comparison bar chart
ax = axes[1]
strat_names = list(strategies.keys())
means = [cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
         for cv in strategies.values()]
stds  = [cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1).std()
         for cv in strategies.values()]
ax.barh(strat_names, means, xerr=stds, color='#1976D2', alpha=0.8, edgecolor='white', capsize=4)
ax.set_xlabel('Mean AUC ± Std')
ax.set_title('CV Strategy Comparison\n(more repeats = lower std)')
ax.set_xlim(0.97, 1.00); ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Nested CV: unbiased estimate when tuning hyperparameters ──────
from sklearn.model_selection import GridSearchCV
print('\n=== Nested CV (unbiased estimate with hyperparameter tuning) ===')
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
param_grid = {'model__n_estimators': [50, 100], 'model__max_depth': [None, 5, 10]}
gs = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='roc_auc', n_jobs=-1)
nested_scores = cross_val_score(gs, X, y, cv=outer_cv, scoring='roc_auc', n_jobs=-1)
print(f'Nested CV AUC: {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print('(This is unbiased even though we tuned hyperparameters)')
```